# E-commerce data cleaning walkthrough

This notebook shows the raw data, the problems in it, and how `cleaning/clean_data.py` fixes each one.
Run it from the project root after `python scripts/generate_sample_data.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "cleaning" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from cleaning.clean_data import clean_all, load_raw, write_report

## 1. Look at the raw data

In [ ]:
customers_raw, orders_raw = load_raw()
print("customers:", customers_raw.shape, "| orders:", orders_raw.shape)
customers_raw.head(8)

In [ ]:
orders_raw.head(8)

## 2. Problems visible in the raw files

- duplicated rows
- names in ALL CAPS, lower case and with extra spaces
- emails that are blank or malformed
- dates in 3+ formats
- prices with `$` and thousands separators
- orders pointing at customers who do not exist
- statuses written as `Completed`, `COMPLETE`, `completed`

In [ ]:
print("duplicate customer ids:", customers_raw["customer_id"].duplicated().sum())
print("duplicate order ids   :", orders_raw["order_id"].duplicated().sum())
print("blank emails          :", (customers_raw["email"].str.strip() == "").sum())
print("status values         :", sorted(orders_raw["status"].unique())[:10])

## 3. Apply the cleaning functions

In [ ]:
customers, orders, customer_issues, order_issues = clean_all()
print("clean customers:", customers.shape)
print("clean orders   :", orders.shape)
customers.head()

In [ ]:
orders.head()

## 4. Save the cleaning report

Every removed row is counted so the cleaning is auditable.

In [ ]:
report_path = write_report(customer_issues, order_issues)
print(report_path.read_text())

## 5. Next step

Load the cleaned data and run the SQL analysis from the terminal:

```bash
python load_to_db.py
python export_reports.py
```

The business questions (monthly revenue, top products, repeat customer retention) live in `sql/analysis_queries.sql`.